# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Show overview metadata (access attributes, not as dict)
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets and their field IDs with names and data types.

Each entity is referenced by its `@id`. All references to record sets, fields, and columns use their respective `@id` attributes as specified by the Croissant schema.

In [ ]:
# List all record sets and for each, print its fields (columns) and their ids

record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record set(s).\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}")
        print(f"      Name: {field.name}")
        print(f"      Data type: {field.data_type}")
        print(f"      Description: {field.description}")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing record sets and field `@id`s explicitly as found above.

In [ ]:
# Extract data from every record set
dataframes = {}
for rset in record_sets:
    records = list(dataset.records(record_set=rset.id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rset.id] = df
        print(f"\nLoaded data for RecordSet @id={rset.id}: {len(df)} rows, columns: {list(df.columns)}")
    else:
        print(f"\nRecordSet @id={rset.id} contains no records to load.")
        dataframes[rset.id] = pd.DataFrame()

# For demonstration, pick the first non-empty record set
chosen_rs = None
for rset_id, df in dataframes.items():
    if not df.empty:
        chosen_rs = rset_id
        print(f"\nFields (columns) for RecordSet @id={rset_id}:")
        print(df.columns.tolist())
        display(df.head())
        break
if chosen_rs is None:
    print("No data available in any record set to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, and grouping.

All field references use their `@id` as shown in the Data Overview.

In [ ]:
import numpy as np

# EDA is performed on the first non-empty record set loaded above.
if chosen_rs is not None:
    df = dataframes[chosen_rs]

    # Try to find a candidate numeric field by inspecting record set definition or DataFrame dtypes
    numeric_field = None
    group_field = None
    # Attempt to select numeric and group fields by looking at datatypes and names
    for col in df.columns:
        # Heuristically pick an integer/float column
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break
    
    # Try to use a string/categorical column as group field
    for col in df.columns:
        if numeric_field and col != numeric_field and df[col].dtype == object:
            group_field = col
            break

    if numeric_field:
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records in RecordSet @id={chosen_rs} with {numeric_field} > {threshold} (mean):")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_field = f"{numeric_field}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} (field `@id`: {numeric_field}) for filtered records:")
        display(filtered_df[[numeric_field, norm_field]].head())

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by field `@id`: {group_field} and computed mean of {numeric_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for this RecordSet; please check the columns manually.")
else:
    print("No record set selected; please verify previous steps.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All field references are by `@id` as before.

Below, we use matplotlib and seaborn for plotting. Adapt field references to your use-case as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_rs is not None and numeric_field is not None:
    plt.figure(figsize=(7, 4))
    # Distribution plot
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of field @id: {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group_field available, boxplot
    if group_field:
        plt.figure(figsize=(12, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field or group selected for visualization.")

## 6. Conclusion
In this notebook, we successfully loaded and explored the FAIR^2 dataset for mass spectrometry analysis using `mlcroissant`. 

We reviewed the available record sets and fields (referencing entities by their `@id`s), extracted the data, applied common EDA operations (filtering, normalization, grouping), and visualized data distributions. 

To further analyze or prepare this data for ML or downstream research, repeat or adapt the steps above for the record sets and fields of interest. Always reference fields and record sets by their `@id` as defined in the dataset's Croissant schema.